# Mini-Projeto Avaliativo - Análise de Dados com Python [T8]
**Aluno:** Lucas Benevides  
**Turma:** T8 - Módulo 1 - Semana 07  
**Tema:** Análise Exploratória de Dados (AED) no Varejo  
**Base de Dados:** Kaggle - [Base Varejo](https://www.kaggle.com/datasets/namespaiva/base-varejo/data)


## 1. Contexto e Unidade da Análise
Este projeto realiza uma Análise Exploratória de Dados (AED) completa sobre uma base de transações varejistas.

> **Definição da Unidade de Análise:**  
> Cada linha da base representa um **item comprado** associado a uma transação (`CO_ID`), cliente (`CL_ID`) e produto (`PR_ID`). Portanto, a contagem de registros reflete o total de itens registrados, e não o número de compras ou de clientes distintos.


## 2. Sprint 1: Importação e Diagnóstico de Qualidade
Leitura da base com delimitador ponto e vírgula (`;`), diagnóstico de estrutura, verificação de nulos, busca por strings vazias e identificação de inconsistências.


In [ ]:
import pandas as pd
import numpy as np
import os

# Localização flexível da base
caminhos = [
    os.path.join('data', 'Base Varejo.csv'),
    'Base Varejo.csv',
    os.path.join('..', 'Análise de Dados com Python [T8]', 'Base Varejo.csv', 'Base Varejo.csv')
]
caminho_base = next((c for c in caminhos if os.path.exists(c)), 'Base Varejo.csv')

# Leitura com delimitador correto
df_bruto = pd.read_csv(caminho_base, sep=';', low_memory=False)

print(f'Total de Linhas: {df_bruto.shape[0]:,}')
print(f'Total de Colunas: {df_bruto.shape[1]}')
df_bruto.head()


In [ ]:
# Tipos de dados e informações estruturais
df_bruto.info()


In [ ]:
# Diagnóstico de valores ausentes (NaN) por coluna
print('Valores ausentes por coluna:')
print(df_bruto.isna().sum())


In [ ]:
# Diagnóstico de strings vazias ou compostas apenas por espaços
print('Contagem de textos vazios:')
for col in df_bruto.select_dtypes(include='object').columns:
    vazios = df_bruto[col].astype(str).str.strip().eq('').sum()
    print(f'  {col}: {vazios} vazios')


In [ ]:
# Diagnóstico de duplicatas e inconsistências textuais
duplicatas_iniciais = df_bruto.duplicated().sum()
nd_cat = (df_bruto['PR_CAT'] == '#N/D').sum()
nd_nome = (df_bruto['PR_NOME'] == '#N/D').sum()

print(f'Linhas duplicadas (todas as colunas): {duplicatas_iniciais:,}')
print(f'Ocorrências de "#N/D" em PR_CAT: {nd_cat:,}')
print(f'Ocorrências de "#N/D" em PR_NOME: {nd_nome:,}')


## 3. Sprint 2 & 3: Funções Reutilizáveis e Limpeza Justificada
Criação de funções com parâmetros e retorno para padronização e resumo estatístico, seguida pelo tratamento justificado da base.


In [ ]:
def padronizar_texto(valor):
    """
    Padroniza valores de texto:
    - Trata valores nulos
    - Remove espaços nas extremidades (.strip())
    - Converte inconsistências '#N/D' para 'NAO IDENTIFICADO'
    - Normaliza caracteres e caixa alta
    """
    if pd.isna(valor):
        return valor
    texto = str(valor).strip()
    if texto.upper() in ['#N/D', '#NA', 'NAN', 'NULL']:
        return 'NAO IDENTIFICADO'
    texto = texto.upper()
    texto = texto.replace('REFRIGERANTE LIMaO', 'REFRIGERANTE LIMAO')
    return texto

def resumo_numerico(serie):
    """
    Calcula e retorna estatísticas descritivas básicas de uma série numérica:
    Contagem de válidos, média, mínimo e máximo.
    """
    return {
        'contagem': int(serie.count()),
        'media': float(serie.mean()),
        'minimo': float(serie.min()),
        'maximo': float(serie.max())
    }


In [ ]:
# Execução da Limpeza Justificada
df_tratado = df_bruto.copy()

# 1. Remoção de colunas residuais causadas por delimitadores extras no CSV
colunas_remover = [c for c in df_tratado.columns if c.startswith('Unnamed')]
df_tratado = df_tratado.drop(columns=colunas_remover)

# 2. Padronização textual de colunas categóricas
df_tratado['PR_CAT'] = df_tratado['PR_CAT'].apply(padronizar_texto)
df_tratado['PR_NOME'] = df_tratado['PR_NOME'].apply(padronizar_texto)

# 3. Tratamento de duplicatas confirmadas
linhas_antes = len(df_tratado)
df_tratado = df_tratado.drop_duplicates()
linhas_depois = len(df_tratado)

print(f'Colunas removidas: {colunas_remover}')
print(f'Linhas duplicadas removidas: {linhas_antes - linhas_depois:,}')
print(f'Shape antes : {df_bruto.shape}')
print(f'Shape depois: {df_tratado.shape}')


## 4. Sprint 4: Análise Exploratória (Estatísticas, Filtro e Agrupamentos)
Análise fundamentada nas regras de negócio da base de varejo.


In [ ]:
# 1. Estatísticas descritivas da coluna numérica 'CL_FHL' (Número de Filhos)
# Obs: IDs (CO_ID, CL_ID, PR_ID) não possuem significado como métricas contínuas.
stats_filhos = resumo_numerico(df_tratado['CL_FHL'])
print('Resumo Numérico da Variável CL_FHL (Número de Filhos):')
for k, v in stats_filhos.items():
    print(f'  {k}: {v}')


In [ ]:
# 2. Filtro com significado de negócio: Clientes com dependentes (CL_FHL > 0)
df_familias = df_tratado[df_tratado['CL_FHL'] > 0]
perc_familias = (len(df_familias) / len(df_tratado)) * 100

print(f'Itens comprados por clientes com filhos: {len(df_familias):,} ({perc_familias:.2f}% do total)')


In [ ]:
# 3. Agrupamento 1: Volume de itens por Categoria de Produto
itens_por_categoria = df_tratado.groupby('PR_CAT').size().sort_values(ascending=False)
print('Volume de Itens por Categoria:')
print(itens_por_categoria)


In [ ]:
# 4. Agrupamento 2: Média de filhos por Segmento de Cliente (CL_SEG)
media_filhos_segmento = df_tratado.groupby('CL_SEG')['CL_FHL'].mean().round(2)
print('Média de Filhos por Segmento de Cliente:')
print(media_filhos_segmento)


## 5. Sprint 5: Exportação da Base Tratada
Exportação sem exportar o índice (`index=False`) conforme exigido no regulamento.


In [ ]:
caminho_exportacao = 'varejo_limpo.csv'
df_tratado.to_csv(caminho_exportacao, index=False)
print(f'Base tratada exportada com sucesso em: {caminho_exportacao}')
print(f'Tamanho do arquivo: {os.path.getsize(caminho_exportacao)/(1024*1024):.2f} MB')


## 6. Conclusões e Comunicação dos Resultados

### Conclusões Numéricas da Análise:
1. **Predomínio da Categoria Alimentos:** A categoria `ALIMENTOS` concentrou **384.197 itens** vendidos (**52,38%** da base tratada), sendo o motor principal de fluxo no varejo analisado.
2. **Relevância de Higiene e Limpeza:** Juntas, as categorias `HIGIENE` (**137.702 itens**) e `LIMPEZA` (**128.632 itens**) perfazem **36,31%** das vendas, consolidando a vocação de cesta básica do estabelecimento.
3. **Perfil Familiar dos Consumidores:** A média de filhos por cliente é de **1,15**, sendo que **47,51%** dos registros de compra correspondem a clientes que possuem ao menos 1 filho.
4. **Homogeneidade entre Segmentos:** A média de filhos varia de forma muito sutil entre os segmentos de clientes (Segmento A: 1,04; Segmento B: 1,15; Segmento C: 1,18).

### Recomendação de Negócio:
Desenvolver estratégias promocionais de cross-selling entre as categorias **Alimentos** e **Higiene/Limpeza** voltadas para o público familiar (ex: combos para lanches infantis e higiene pessoal), aproveitando que praticamente metade da base é composta por famílias com dependentes.

### Limitação da Análise:
A base de dados não dispõe de atributos monetários (como preço de venda, custo ou faturamento) e nem coluna explícita de quantidade por linha. Dessa forma, as análises estão estritamente restritas a volumes de itens movimentados e perfis demográficos, não sendo viável mensurar faturamento total ou ticket médio.
